# 04 — Analyse & Erreurs
Charge le benchmark clean, génère les visualisations d'analyse approfondie.
Prérequis : avoir lancé `03_experiments.ipynb`.

In [5]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

BASE         = r'C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten'
CLEAN_PROC   = os.path.join(BASE, 'data', 'processed_clean')
CLEAN_MODELS = os.path.join(BASE, 'models', 'clean')

y_test = np.load(os.path.join(CLEAN_PROC, 'y_test.npy'))
y_val  = np.load(os.path.join(CLEAN_PROC, 'y_val.npy'))

with open(os.path.join(CLEAN_PROC, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)
class_names = [str(c) for c in le.inverse_transform(range(27))]

test_split = pd.read_csv(os.path.join(CLEAN_PROC, 'test.csv'))
print('Setup OK')

Setup OK


In [6]:
bench_path = os.path.join(CLEAN_MODELS, 'benchmark_clean.csv')
df_bench = pd.read_csv(bench_path)
print(df_bench[['name', 'F1_val', 'F1_test', 'Acc_test']].to_string(index=False))

                        name  F1_val  F1_test  Acc_test
     E2_dino_lin_meta_logreg  0.8584   0.8541    0.8607
E3_dino_lin_crop_meta_logreg  0.8589   0.8539    0.8605
   E4_dino_lin_crop_meta_mlp  0.8451   0.8447    0.8533
                E1_text_only  0.8340   0.8343    0.8430
         E5_dino_ft_meta_mlp  0.7346   0.7330    0.7528


## Matrices de confusion — Meilleur modèle vs Baseline texte

In [7]:
import numpy as np

P_text_test   = np.load(os.path.join(CLEAN_PROC, 'P_text_test.npy'))
P_ctfidf_test = np.load(os.path.join(CLEAN_PROC, 'X_ctfidf_test.npy'))
X_extra_test  = np.load(os.path.join(CLEAN_PROC, 'X_extra_test.npy'))

# Probas image par source (chargées à la demande selon l'expérience)
IMG_PROBAS_TEST = {
    'lin'     : np.load(os.path.join(CLEAN_PROC, 'P_dino_lin_test.npy')),
    'lin_crop': np.load(os.path.join(CLEAN_PROC, 'P_dino_lin_crop_test.npy')),
    'ft'      : np.load(os.path.join(CLEAN_PROC, 'P_dino_ft_test.npy')),
}

# Config minimale pour reconstruire les features de chaque expérience
EXP_META_CFG = {
    'E1_text_only'               : {'img': None,       'rich': False, 'extra': False},
    'E2_dino_lin_meta_logreg'    : {'img': 'lin',      'rich': False, 'extra': False},
    'E3_dino_lin_crop_meta_logreg': {'img': 'lin_crop','rich': False, 'extra': False},
    'E4_dino_lin_crop_meta_mlp'  : {'img': 'lin_crop', 'rich': True,  'extra': True},
    'E5_dino_ft_meta_mlp'        : {'img': 'ft',       'rich': True,  'extra': True},
}

def entropy(P):
    P = np.clip(P, 1e-10, 1.0)
    return -(P * np.log(P)).sum(axis=1, keepdims=True).astype('float32')

def margin(P):
    s = np.sort(P, axis=1)
    return (s[:, -1] - s[:, -2]).reshape(-1, 1).astype('float32')

def build_meta(P_txt, P_img, ctfidf, extra=None, rich=True):
    feats = [P_txt, P_img, ctfidf]
    if rich:
        feats += [
            entropy(P_txt), entropy(P_img),
            margin(P_txt),  margin(P_img),
            P_txt.max(axis=1, keepdims=True),
            P_img.max(axis=1, keepdims=True),
            (P_txt.argmax(1) == P_img.argmax(1)).astype('float32').reshape(-1, 1),
        ]
    if extra is not None:
        feats.append(extra.astype('float32'))
    return np.hstack(feats).astype('float32')

# Charger le meilleur meta-classifieur et reconstruire ses features correctement
best_name = df_bench.iloc[0]['name']
best_ckpt = os.path.join(CLEAN_MODELS, f'meta_{best_name}.pkl')
cfg = EXP_META_CFG.get(best_name, {'img': 'lin', 'rich': False, 'extra': False})

if os.path.exists(best_ckpt) and cfg['img'] is not None:
    with open(best_ckpt, 'rb') as f: best_meta = pickle.load(f)
    P_img_test = IMG_PROBAS_TEST[cfg['img']]
    xte = X_extra_test if cfg['extra'] else None
    Xm_test = build_meta(P_text_test, P_img_test, P_ctfidf_test, xte, cfg['rich'])
    assert Xm_test.shape[1] == best_meta.n_features_in_, \
        f"Feature mismatch : {Xm_test.shape[1]} vs {best_meta.n_features_in_} attendus"
    preds_best = best_meta.predict(Xm_test)
    print(f'Features meta : {Xm_test.shape[1]} | img_source={cfg["img"]} rich={cfg["rich"]} extra={cfg["extra"]}')
else:
    preds_best = P_text_test.argmax(1)
    print(f'Fallback : text-only (pas de ckpt meta ou exp sans image)')

preds_text = P_text_test.argmax(1)
print(f'Meilleur ({best_name})  : F1={f1_score(y_test, preds_best, average="macro"):.4f}')
print(f'Baseline texte seul      : F1={f1_score(y_test, preds_text, average="macro"):.4f}')

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten\\data\\processed_clean\\P_dino_ft_test.npy'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
short = [c[:10] for c in class_names]
for ax, (preds, title) in zip(axes, [
    (preds_text, f'Baseline text-only\nF1={f1_score(y_test, preds_text, average="macro"):.4f}'),
    (preds_best, f'{best_name}\nF1={f1_score(y_test, preds_best, average="macro"):.4f}'),
]):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=short).plot(ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(title, fontsize=10)
plt.suptitle('Matrices de confusion — Baseline vs Meilleur modèle', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CLEAN_MODELS, 'confusion_matrices.png'), dpi=110)
plt.show()

## Analyse des erreurs — classes difficiles

In [ ]:
# F1 par classe : tous les modeles
f1_text = f1_score(y_test, preds_text, average=None)
f1_best = f1_score(y_test, preds_best, average=None)

df_cls = pd.DataFrame({
    'Classe'     : class_names,
    'F1_text'    : f1_text,
    'F1_best'    : f1_best,
    'Delta'      : f1_best - f1_text,
    'count_test' : [(y_test == i).sum() for i in range(27)],
}).sort_values('F1_best')

print('Classes les plus difficiles (F1_best < 0.7) :')
print(df_cls[df_cls['F1_best'] < 0.7][['Classe','F1_text','F1_best','Delta','count_test']].to_string(index=False))
print()
print('Plus grands gains (delta > 0.03) :')
print(df_cls[df_cls['Delta'] > 0.03][['Classe','F1_text','F1_best','Delta']].sort_values('Delta', ascending=False).to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# F1 par classe : texte vs meilleur
df_plot = df_cls.sort_values('F1_best')
y_pos = range(len(df_plot))
ax1.barh(y_pos, df_plot['F1_text'], alpha=0.5, label='Text only', color='steelblue')
ax1.barh(y_pos, df_plot['F1_best'], alpha=0.7, label=best_name, color='#2ECC71')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(df_plot['Classe'], fontsize=8)
ax1.axvline(f1_text.mean(), color='steelblue', linestyle=':', linewidth=1)
ax1.axvline(f1_best.mean(), color='#27AE60',   linestyle=':', linewidth=1)
ax1.set_xlabel('F1')
ax1.set_title('F1 par classe')
ax1.legend()

# Delta par classe
colors_d = ['#2ECC71' if d >= 0 else '#E74C3C' for d in df_plot['Delta']]
ax2.barh(y_pos, df_plot['Delta'], color=colors_d, alpha=0.85)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(df_plot['Classe'], fontsize=8)
ax2.set_xlabel('Delta F1 (meilleur - baseline)')
ax2.set_title('Gain par classe')

plt.suptitle(f'Analyse par classe : {best_name} vs Text-only', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(CLEAN_MODELS, 'f1_per_class_delta.png'), dpi=120)
plt.show()

## Positionnement vs SIGIR eCommerce Challenge 2020

In [ ]:
# ── Positionnement vs SIGIR eCommerce Challenge 2020 ─────────────────
# Challenge organisé en 2020, 16 équipes, dataset Rakuten France (84k produits, 27 classes)

SIGIR_REFS = [
    # (label, F1, couleur, annotation)
    ('Rakuten\nbaseline',          0.8705, '#95A5A6', 'Baseline officiel\n(Rakuten, 2020)'),
    ('Synerise\n(5ème place)',     0.8972, '#F39C12', 'EfficientNet + CamemBERT\n(SIGIR eCom\'20, paper #3)'),
    ('Wang-Shuo\n(1ère place)',    0.9194, '#E74C3C', 'Decision-Level Fusion\n(meilleur score challenge)'),
    ('depshad\n(3ème place)',      0.9194, '#C0392B', 'Deep Multi-level Boosted Fusion\n(SIGIR eCom\'20, paper #8)'),
]

# Ajouter nos experiences depuis le benchmark
our_results = [(row['name'].replace('_', '\n'), row['F1_test'], '#3498DB', f"Notre exp\nF1={row['F1_test']:.4f}")
               for _, row in df_bench.iterrows()]

all_bars = our_results + SIGIR_REFS
labels  = [x[0] for x in all_bars]
scores  = [x[1] for x in all_bars]
colors  = [x[2] for x in all_bars]
annots  = [x[3] for x in all_bars]

fig, ax = plt.subplots(figsize=(max(14, len(all_bars) * 1.4), 7))

bars = ax.bar(range(len(all_bars)), scores, color=colors, alpha=0.85, width=0.6, edgecolor='white', linewidth=0.8)

# Ligne de référence : notre meilleur
our_best = df_bench.iloc[0]['F1_test']
ax.axhline(our_best, color='#3498DB', linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Notre meilleur ({our_best:.4f})')

# Lignes des références SIGIR
ax.axhline(0.8705, color='#95A5A6', linestyle=':', linewidth=1, alpha=0.7, label='Baseline Rakuten (0.8705)')
ax.axhline(0.9194, color='#E74C3C', linestyle=':', linewidth=1, alpha=0.7, label='SIGIR 1ère place (0.9194)')

# Valeurs au-dessus des barres
for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(bar.get_x() + bar.get_width()/2, score + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Séparateur nos résultats / challenge
n_ours = len(our_results)
ax.axvline(n_ours - 0.5, color='black', linewidth=1.5, linestyle='-', alpha=0.3)
ax.text(n_ours - 0.5, ax.get_ylim()[0] + 0.001, '  ← nos résultats | SIGIR 2020 →',
        fontsize=8, color='gray', va='bottom')

ax.set_xticks(range(len(all_bars)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('F1 macro (test)', fontsize=11)
ax.set_ylim(min(scores) - 0.02, max(scores) + 0.025)
ax.set_title('Benchmark clean vs SIGIR eCommerce Challenge 2020\n(16 équipes, dataset Rakuten France, 27 classes)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Écart depuis notre meilleur jusqu'au 1er du challenge
gap = 0.9194 - our_best
ax.annotate('', xy=(n_ours + len(SIGIR_REFS) - 1, 0.9194),
            xytext=(n_ours + len(SIGIR_REFS) - 1, our_best),
            arrowprops=dict(arrowstyle='<->', color='#E74C3C', lw=1.5))
ax.text(n_ours + len(SIGIR_REFS) - 0.55, (0.9194 + our_best) / 2,
        f'  Δ={gap:.4f}', color='#E74C3C', fontsize=9, va='center')

plt.tight_layout()
plt.savefig(os.path.join(CLEAN_MODELS, 'sigir_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'\nÉcart vers le 1er du challenge : {gap:.4f} ({gap*100:.2f} points F1)')
print(f'Écart vers le baseline Rakuten  : {our_best - 0.8705:+.4f}')

## Interprétabilité — Analyse exploratoire du dataset

In [ ]:
# ── 1. Top mots discriminants par classe (coefficients LogReg TF-IDF) ─
import scipy.sparse as sp

tfidf_path = os.path.join(CLEAN_MODELS, 'tfidf_vectorizer.pkl')
clf_path   = os.path.join(CLEAN_MODELS, 'clf_text_full.pkl')

if not os.path.exists(tfidf_path) or not os.path.exists(clf_path):
    print('Vectorizer ou clf_text_full introuvable — lancer 02_level1_probas.ipynb d\'abord.')
else:
    with open(tfidf_path, 'rb') as f: tfidf_vec = pickle.load(f)
    with open(clf_path,   'rb') as f: clf_text  = pickle.load(f)

    vocab = np.array(tfidf_vec.get_feature_names_out())
    coef  = clf_text.coef_   # (27, vocab_size)
    TOP_N = 8

    fig, axes = plt.subplots(6, 5, figsize=(22, 20))
    axes = axes.flatten()
    for lbl, name in enumerate(class_names):
        top_idx  = np.argsort(coef[lbl])[-TOP_N:][::-1]
        bot_idx  = np.argsort(coef[lbl])[:TOP_N]
        ax = axes[lbl]
        # positifs (pour cette classe) en vert, négatifs en rouge
        vals  = np.concatenate([coef[lbl][top_idx], coef[lbl][bot_idx]])
        words = np.concatenate([vocab[top_idx], vocab[bot_idx]])
        colors_bar = ['#2ECC71'] * TOP_N + ['#E74C3C'] * TOP_N
        y_pos = range(len(vals))
        ax.barh(y_pos, vals[::-1], color=colors_bar[::-1], alpha=0.85)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words[::-1], fontsize=6)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.set_title(f'{name}', fontsize=7, fontweight='bold')
        ax.tick_params(axis='x', labelsize=6)

    for i in range(27, len(axes)): axes[i].axis('off')
    plt.suptitle('Interprétabilité LogReg TF-IDF — Top/Bottom 8 mots par classe\n'
                 '(vert = pousse vers cette classe | rouge = pousse contre)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CLEAN_MODELS, 'interpret_tfidf_coef.png'), dpi=110, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 2. Cartes d'attention DINOv2 (ViT attention maps) ─────────────────
# DINOv2-base : ViT-B/14 → patches 14×14px → 16×16=256 patches pour 224×224
# On visualise l'attention du token [CLS] vers les patches (dernier bloc)
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from transformers import AutoModel

SPLIT_DIR = r'C:/Users/veyss/Nouveau dossier/nov25_alt1_mle_rakuten/data/split_no_crop'
# Nouveau checkpoint (entraînement amélioré depuis 02_level1_probas)
FT_CKPT = os.path.join(CLEAN_MODELS, 'dino_ft_full.pth')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone = AutoModel.from_pretrained('facebook/dinov2-base').to(device)

# Architecture GELU pour correspondre au nouveau DINOv2Classifier
class DINOv2Clf(nn.Module):
    def __init__(self, bb):
        super().__init__()
        self.backbone = bb
        self.head = nn.Sequential(
            nn.Linear(768, 256), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(256, 27)
        )
    def forward(self, pixel_values):
        return self.head(self.backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :])

if os.path.exists(FT_CKPT):
    clf_dino = DINOv2Clf(backbone).to(device)
    clf_dino.load_state_dict(torch.load(FT_CKPT, map_location=device))
    clf_dino.eval()
    model_for_attn = clf_dino.backbone
    print(f'Modèle FT chargé : {os.path.basename(FT_CKPT)}')
else:
    model_for_attn = backbone.eval()
    print('Backbone gelé (pas de FT ckpt — relancer 02_level1_probas.ipynb)')

transform = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def get_attention_map(img_path):
    """Retourne (image PIL 224×224, carte attention [CLS→patches] 16×16)."""
    img_pil = Image.open(img_path).convert('RGB')
    x = transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model_for_attn(pixel_values=x, output_attentions=True)
    attn = out.attentions[-1][0]       # (n_heads, 257, 257)
    cls_attn = attn[:, 0, 1:]         # [CLS] → patches : (n_heads, 256)
    cls_attn = cls_attn.mean(0)        # moyenne têtes : (256,)
    cls_attn = cls_attn.reshape(16, 16).cpu().numpy()
    cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
    return img_pil.resize((224, 224)), cls_attn

# 3 erreurs + 3 corrects
errors_idx   = np.where(y_test != preds_best)[0][:50]
corrects_idx = np.where(y_test == preds_best)[0][:50]
sample_idx   = list(errors_idx[:3]) + list(corrects_idx[:3])

n_show = len(sample_idx)
fig, axes = plt.subplots(n_show, 3, figsize=(13, n_show * 3.2))

for row, idx in enumerate(sample_idx):
    img_name = test_split.iloc[idx]['image_name']
    img_path = os.path.join(SPLIT_DIR, 'test', img_name)
    if not os.path.exists(img_path):
        for ax in axes[row]: ax.axis('off')
        continue

    img_pil, attn_map = get_attention_map(img_path)
    true_cls = class_names[y_test[idx]]
    pred_cls = class_names[preds_best[idx]]
    correct  = y_test[idx] == preds_best[idx]
    color    = '#2ECC71' if correct else '#E74C3C'
    status   = 'Correct' if correct else 'Erreur'

    axes[row, 0].imshow(img_pil)
    axes[row, 0].set_title(f'[{status}] Vrai: {true_cls}', fontsize=8, color=color)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(attn_map, cmap='inferno', interpolation='bilinear')
    axes[row, 1].set_title('Attention [CLS] → patches', fontsize=8)
    axes[row, 1].axis('off')

    attn_resized = np.array(Image.fromarray((attn_map * 255).astype('uint8')).resize(
        (224, 224), Image.BILINEAR)) / 255.0
    img_arr = np.array(img_pil) / 255.0
    overlay = img_arr * 0.5 + plt.cm.inferno(attn_resized)[:, :, :3] * 0.5
    axes[row, 2].imshow(np.clip(overlay, 0, 1))
    axes[row, 2].set_title(f'Overlay | Prédit: {pred_cls}', fontsize=8, color=color)
    axes[row, 2].axis('off')

plt.suptitle("Interprétabilité DINOv2 — Cartes d'attention [CLS] (équivalent Grad-CAM pour ViT)\n"
             "Rouge = zones les plus regardées par le modèle",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CLEAN_MODELS, 'interpret_dino_attention.png'), dpi=110, bbox_inches='tight')
plt.show()

## Résumé global

In [ ]:
prev_best = 0.8550
print('\n' + '='*70)
print('RESUME FINAL — BENCHMARK CLEAN')
print('='*70)
for _, row in df_bench.iterrows():
    flag = ' *** NOUVEAU RECORD' if row['F1_test'] > prev_best else ''
    print(f"  {row['name']:<30} test F1={row['F1_test']:.4f}{flag}")
print('='*70)
print(f'Precedent meilleur (notebook 03 FT)     : {prev_best:.4f}')
print(f'Nouveau meilleur                         : {df_bench.iloc[0]["F1_test"]:.4f}')
above = (df_bench['F1_test'] > prev_best).sum()
print(f'{above}/5 experiences battent lancien record')
print()
print('Fichiers generes dans models/clean/ :')
for f in sorted(os.listdir(CLEAN_MODELS)):
    sz = os.path.getsize(os.path.join(CLEAN_MODELS, f)) // 1024
    print(f'  {f}  ({sz} KB)')